<a href="https://colab.research.google.com/github/AlexeyProvorov/Other_projects/blob/master/W3_S6_LC1_OpenAI_API_Advanced_Building_Chatbots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><a target="_blank" href="https://academy.constructor.org/"><img src=https://lh3.googleusercontent.com/d/1EmH3Jks5CpJy0zK3JbkvJZkeqWtVcxhB width="500" style="background:none; border:none; box-shadow:none;" /></a> </center>
<hr />

# <h1 align="center">Live Coding - Session 6</h1> </center>
<center> <h1>Build Chatbots with OpenAI's API</h1>

<hr />
<center>Constructor Academy, 2025</center>

In this notebook we will showcase a few demos where you will learn how to:

- Use the OpenAI API for GPT4o-mini model
- Learn about key message types in OpenAI's API
- How to have a conversation using OpenAI's API
- Learn how to build a Text-based Chatbot using a GPT4o-mini model
- Learn how to build a UI-based Chatbot Application using a GPT4o-mini model and Gradio

## Use UTF-8 encoding for text processing, overriding the system's default locale encoding.

In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

# Load the API key using Google `Secrects`

In [ ]:
from google.colab import userdata
import openai

openai.api_key = userdata.get('openai_key')


## Message Types in ChatGPT - CHAT COMPLETIONS

To have a more interactive and dynamic conversation with LLMs, you can use messages in the GPT4o model instead of the old prompt-style using with completions.


Here's how it works:

- Instead of sending a single string as your prompt, you send a list of messages as your input.

- Each message in the list has two properties: `role` and `content`.

  - The `'role'` can take one of four values: `'system'`, `'user'`, `'assistant'` or `'tool'`

  - The `'content'` contains the text of the message from the `role`.

- The `system` instruction can give high level instructions for the conversation

- The messages are processed in the order they appear in the list, and the assistant responds accordingly.

In [ ]:
message_history =  [
    {'role':'user', 'content':'hello there'}
]

In [ ]:
response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=message_history,
        temperature=0.1,
    )
response

ChatCompletion(id='chatcmpl-CVjl3Ximy5Wiq3rSwwMpUE9eTfhgJ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1761680097, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_560af6e559', usage=CompletionUsage(completion_tokens=9, prompt_tokens=9, total_tokens=18, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [ ]:
chatgpt_response = response.choices[0].message
chatgpt_response

ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)

In [ ]:
type(response)

openai.types.chat.chat_completion.ChatCompletion

### ChatCompletionMessage Object

The `ChatCompletionMessage` is a custom object type returned by the OpenAI API when we request a chat completion. It is based on [Pydantic](https://docs.pydantic.dev/latest/why/#performance)'s base model, which means it comes with built-in methods for data validation and serialization.


## OPEN AI - ChatCompletion API does not remember previous conversations unless stored

In [ ]:
message_history =  [
    {'role':'user', 'content':'hello my name is Matteo'}
]

response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=message_history,
        temperature=0, # degree of randomness of the model's output
    )

response.choices[0].message.content

'Hello Matteo! How can I assist you today?'

In [ ]:
message_history =  [
    {'role':'user', 'content':'hello do you remember my name?'}
]

response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=message_history,
        temperature=0, # degree of randomness of the model's output
    )

response.choices[0].message.content

"I don’t have the ability to remember personal information or past interactions. But I'm here to help you with anything you need! What can I assist you with today?"

In [ ]:
def transform_message(message):
    """
    Transforms a ChatCompletionMessage object to a dictionary with just 'role' and 'content' keys.

    Parameters:
    message (ChatCompletionMessage): The message object to transform.

    Returns:
    dict: A dictionary with 'role' and 'content' keys.
    """
    return {
        'role': message.role,
        'content': message.content
    }

## Storing conversation history and sending to OpenAI's API helps it remember context

In [ ]:
message_history =  [
    {'role':'user', 'content':'hello my name is Matteo'}
]

response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=message_history,
        temperature=0, # degree of randomness of the model's output
    )

#here we are taking the response from GPT and appending to our message_history
gpt_response = transform_message(response.choices[0].message)
message_history.append(gpt_response)

#now we add an extra message from the user
user_message = {'role':'user', 'content':'hello do you remember my name?'}
message_history.append(user_message)

#that's our full prompt which incorporates full message history
message_history

[{'role': 'user', 'content': 'hello my name is Matteo'},
 {'role': 'assistant', 'content': 'Hello Matteo! How can I assist you today?'},
 {'role': 'user', 'content': 'hello do you remember my name?'}]

In [ ]:
#let's send this prompt to GPT
response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=message_history,
        temperature=2, # degree of randomness of the model's output
    )
response

ChatCompletion(id='chatcmpl-CVjpEHtlByuDpSQsUbAXglotkRZCY', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Yes, you mentioned your name is Matteo. How can I help you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1761680356, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_560af6e559', usage=CompletionUsage(completion_tokens=16, prompt_tokens=37, total_tokens=53, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [ ]:
response.choices[0].message.content

'Yes, you mentioned your name is Matteo. How can I help you today?'

#Simulating a conversation with a GPT4 model

1. Start with some initial system message if you want the GPT4 model to behave in a certain way (optional)
2. Add your user message \ prompt to the list of messages (conversation history)
3. Send this history to the GPT4 model and get its response as an assistant message
4. Add this message to your existing conversation history
5. Add a new user message \ prompt to the conversation history
6. Repeat steps 3 - 5 again and again

### Steps 1-4: Adding a system and initial user message. Getting a response assistant message from the API model and adding it to the conversation history

In [ ]:
messages =  [
    {'role':'system', 'content':'Act as a helpful assistant!'},
    {'role':'user', 'content':'Hello'}
]
response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.5, # degree of randomness of the model's output
    )
response = transform_message(response.choices[0].message)
messages.append(response)
messages

[{'role': 'system', 'content': 'Act as a helpful assistant!'},
 {'role': 'user', 'content': 'Hello'},
 {'role': 'assistant', 'content': 'Hello! How can I assist you today?'}]

### Steps 5: Adding a new user message to the conversation history

In [ ]:
messages.append({'role':'user', 'content':'Hi my name is Matteo!'})
messages

[{'role': 'system', 'content': 'Act as a helpful assistant!'},
 {'role': 'user', 'content': 'Hello'},
 {'role': 'assistant', 'content': 'Hello! How can I assist you today?'},
 {'role': 'user', 'content': 'Hi my name is Matteo!'}]

### Steps 6: Repeat previous steps again and again

In [ ]:
response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.5, # degree of randomness of the model's output
)
response = transform_message(response.choices[0].message)
messages.append(response)
messages

[{'role': 'system', 'content': 'Act as a helpful assistant!'},
 {'role': 'user', 'content': 'Hello'},
 {'role': 'assistant', 'content': 'Hello! How can I assist you today?'},
 {'role': 'user', 'content': 'Hi my name is Matteo!'},
 {'role': 'assistant',
  'content': "Hi Matteo! It's great to meet you. How can I help you today?"}]

In [ ]:
messages.append({'role':'user', 'content':'can you tell me a joke?'})
messages

[{'role': 'system', 'content': 'Act as a helpful assistant!'},
 {'role': 'user', 'content': 'Hello'},
 {'role': 'assistant', 'content': 'Hello! How can I assist you today?'},
 {'role': 'user', 'content': 'Hi my name is Matteo!'},
 {'role': 'assistant',
  'content': "Hi Matteo! It's great to meet you. How can I help you today?"},
 {'role': 'user', 'content': 'can you tell me a joke?'}]

In [ ]:
response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.5, # degree of randomness of the model's output
)
response = transform_message(response.choices[0].message)
messages.append(response)
messages

[{'role': 'system', 'content': 'Act as a helpful assistant!'},
 {'role': 'user', 'content': 'Hello'},
 {'role': 'assistant', 'content': 'Hello! How can I assist you today?'},
 {'role': 'user', 'content': 'Hi my name is Matteo!'},
 {'role': 'assistant',
  'content': "Hi Matteo! It's great to meet you. How can I help you today?"},
 {'role': 'user', 'content': 'can you tell me a joke?'},
 {'role': 'assistant',
  'content': 'Of course! Here’s one for you:\n\nWhy did the scarecrow win an award?\n\nBecause he was outstanding in his field! \n\nHope that brought a smile to your face! Would you like to hear another?'}]

In [ ]:
messages.append({'role':'user', 'content':'Do you know my name?'})
messages

[{'role': 'system', 'content': 'Act as a helpful assistant!'},
 {'role': 'user', 'content': 'Hello'},
 {'role': 'assistant', 'content': 'Hello! How can I assist you today?'},
 {'role': 'user', 'content': 'Hi my name is Matteo!'},
 {'role': 'assistant',
  'content': "Hi Matteo! It's great to meet you. How can I help you today?"},
 {'role': 'user', 'content': 'can you tell me a joke?'},
 {'role': 'assistant',
  'content': 'Of course! Here’s one for you:\n\nWhy did the scarecrow win an award?\n\nBecause he was outstanding in his field! \n\nHope that brought a smile to your face! Would you like to hear another?'},
 {'role': 'user', 'content': 'Do you know my name?'}]

In [ ]:
response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.5, # degree of randomness of the model's output
)
response = transform_message(response.choices[0].message)
messages.append(response)
messages

[{'role': 'system', 'content': 'Act as a helpful assistant!'},
 {'role': 'user', 'content': 'Hello'},
 {'role': 'assistant', 'content': 'Hello! How can I assist you today?'},
 {'role': 'user', 'content': 'Hi my name is Matteo!'},
 {'role': 'assistant',
  'content': "Hi Matteo! It's great to meet you. How can I help you today?"},
 {'role': 'user', 'content': 'can you tell me a joke?'},
 {'role': 'assistant',
  'content': 'Of course! Here’s one for you:\n\nWhy did the scarecrow win an award?\n\nBecause he was outstanding in his field! \n\nHope that brought a smile to your face! Would you like to hear another?'},
 {'role': 'user', 'content': 'Do you know my name?'},
 {'role': 'assistant',
  'content': 'Yes, your name is Matteo! How can I assist you further, Matteo?'}]

# Creating and managing a window of conversation history

Having too long a conversation history increases number of tokens, cost and processing time for ChatGPT so often you can keep the last K messages as a window of history.

In [ ]:
# here we have 5 messages
a = ['system_prompt','user1','AI1','user2','AI2','user3']
# if we have a system message we always keep that as it influences the chatgpt behavior
r = a[0]
print(r)
print(a[1:]) # has rest of the conversation messages

system_prompt
['user1', 'AI1', 'user2', 'AI2', 'user3']


In [ ]:
# helps slice the most recent K (here 2) messages from the history
w = 3
a[-w:]

['user2', 'AI2', 'user3']

In [ ]:
# recreate conversation history with system message and the last K (here 2 messages)
a = [r] + a[-w:]
a

['system_prompt', 'user2', 'AI2', 'user3']

## Create Utility Functions for Chatbot

Here we will create utility functions for building a chatbot:

1.   Function to store and add messages to conversation history with a history window
2.   Function to get responses from the GPT4o-mini model
3. Function to take a prompt and add it as a user message to conversation history

### Function to store and add messages to conversation history with a history window

This function takes in new messages and a history of past messages and maintains the conversation history within a specified history window.

In [ ]:
def maintain_chatgpt_message_history(messages, history=None, history_window=4):
    if history is None:
        history = []

    # Simply append the message dictionary
    if isinstance(messages, dict):
        history.append(messages)

    # Keep only the last history_window messages, preserving system message
    if len(history) > history_window:
        system_msg = history[0] if history[0]['role'] == 'system' else None
        history = history[-history_window:]
        if system_msg:
            history.insert(0, system_msg)

    return history

### Function to get response from ChatGPT

This function calls the OpenAI API to get a completion response for the chat based on the message history.

In [ ]:
def get_completion(message_history, model="gpt-4o-mini", temperature=0):

    response = openai.chat.completions.create(
        model=model,
        messages=message_history,
        temperature=temperature,
    )

    # The response.choices[0].message is already a ChatCompletionMessage object
    # which has role and content attributes
    return transform_message(response.choices[0].message)

### Function to take a prompt and add it as a user message to conversation history

This function takes in the current message history and a new user prompt and appends the prompt to the history as a user message

In [ ]:
def add_user_prompt(messages, prompt):
    # Convert single message to list if necessary
    if isinstance(messages, dict):
        messages = [messages]

    # Create a simple object with role and content attributes
    class UserMessage:
        def __init__(self):
            self.role = 'user'
            self.content = prompt

    messages.append(transform_message(UserMessage()))

    return messages

## Trying out Utility Functions for Chatbot

In [ ]:
## start with initial history
message_history = [
    {'role': 'system', 'content': 'Act as a helpful assistant!'},
    {'role': 'user', 'content': 'Hello'},
]
# get response from chatgpt and add to history
response = get_completion(message_history)
message_history = maintain_chatgpt_message_history(messages=response, history=message_history)
message_history

[{'role': 'system', 'content': 'Act as a helpful assistant!'},
 {'role': 'user', 'content': 'Hello'},
 {'role': 'assistant', 'content': 'Hello! How can I assist you today?'}]

In [ ]:
# add new user prompt message
message_history = add_user_prompt(messages=message_history, prompt='Hi! My name is Matteo')
message_history

[{'role': 'system', 'content': 'Act as a helpful assistant!'},
 {'role': 'user', 'content': 'Hello'},
 {'role': 'assistant', 'content': 'Hello! How can I assist you today?'},
 {'role': 'user', 'content': 'Hi! My name is Matteo'}]

In [ ]:
# get response from chatgpt and add to history
response = get_completion(message_history)
message_history = maintain_chatgpt_message_history(messages=response, history=message_history)
message_history

[{'role': 'system', 'content': 'Act as a helpful assistant!'},
 {'role': 'user', 'content': 'Hello'},
 {'role': 'assistant', 'content': 'Hello! How can I assist you today?'},
 {'role': 'user', 'content': 'Hi! My name is Matteo'},
 {'role': 'assistant',
  'content': "Hi Matteo! It's nice to meet you. How can I help you today?"}]

In [ ]:
# add new user prompt message
message_history = add_user_prompt(message_history, prompt='Can you explain AI in 1 line?')
message_history

[{'role': 'system', 'content': 'Act as a helpful assistant!'},
 {'role': 'user', 'content': 'Hello'},
 {'role': 'assistant', 'content': 'Hello! How can I assist you today?'},
 {'role': 'user', 'content': 'Hi! My name is Matteo'},
 {'role': 'assistant',
  'content': "Hi Matteo! It's nice to meet you. How can I help you today?"},
 {'role': 'user', 'content': 'Can you explain AI in 1 line?'}]

In [ ]:
# get response from chatgpt and add to history
response = get_completion(message_history)
message_history = maintain_chatgpt_message_history(messages=response, history=message_history)
message_history

[{'role': 'system', 'content': 'Act as a helpful assistant!'},
 {'role': 'user', 'content': 'Hi! My name is Matteo'},
 {'role': 'assistant',
  'content': "Hi Matteo! It's nice to meet you. How can I help you today?"},
 {'role': 'user', 'content': 'Can you explain AI in 1 line?'},
 {'role': 'assistant',
  'content': 'AI, or artificial intelligence, is the simulation of human intelligence processes by machines, particularly computer systems, enabling them to learn, reason, and make decisions.'}]

# Build a Dummy Text-based Chatbot

---



- This chatbot takes your text input
- It responds back by showing your text

In [ ]:
# This function initializes and runs the dummy chatbot.
def run_dummy_chatbot():

    # This nested function generates a reply from the chatbot. It simply echoes back the user's message.
    def chatbot_reply(message):
        # The function returns the message concatenated with a prefix string.
        return "You said: " + message

    # Print a welcome message when the chatbot starts.
    print("Hello! I am your friendly chatbot. Let's chat! (type 'STOP' to end)")

    # Start an infinite loop to continuously interact with the user.
    while True:
        # Prompt the user for input and store it in a variable.
        prompt = input('User: >>> ')

        # Check if the user's input is 'STOP', indicating they wish to end the conversation.
        # The strip() removes any leading/trailing whitespace, and upper() ensures the check is case-insensitive.
        if prompt.strip().upper() == 'STOP':
            # If the user types 'STOP', print a goodbye message and break out of the loop to end the chat.
            print("Bot: Ciao Ciao!")
            break

        # If the user does not type 'STOP', generate a bot reply using the chatbot_reply function.
        reply = chatbot_reply(prompt)
        # Print the bot's reply to the console.
        print(f"DummyGPT: >>> {reply}")

In [ ]:
# Run the chatbot
run_dummy_chatbot()

Hello! I am your friendly chatbot. Let's chat! (type 'STOP' to end)
User: >>> STOP
Bot: Ciao Ciao!


# Build an Interactive Text-based Chatbot
---



- This chatbot takes your text input
- It stores messages in a conversation history
- Uses this history to get responses from API
- Shows the reply back to you as a chatbot

In [ ]:
# This function initializes and runs a chatbot that uses OpenAI's GPT model to generate replies.
def run_chatgpt_chatbot(system_prompt='', history_window=30, temperature=0.3):

    # This nested function handles the chatbot's reply logic.
    def chatbot_reply(history, prompt):
        # Add the user's prompt to the conversation history.
        history = add_user_prompt(history, prompt)
        # Get a response from the ChatGPT API based on the updated converation history.
        response = get_completion(history, temperature=temperature)
        # Maintain the chat history with the new response and ensure it does not exceed the history window.
        history = maintain_chatgpt_message_history(messages=response, history=history, history_window=history_window)
        # Return the updated history and the content of the chatbot's most recent response.
        return history, response['content']

    # Print a welcome message when the chatbot starts.
    print("Hello! I am your friendly chatbot. Let's chat! (type 'STOP' to end)")

    # Initialize the message history with a system prompt if one is provided, which can set context or instructions.
    if system_prompt:
        message_history = [{"role": "system", "content": system_prompt}]
    else:
        # If there is no system prompt, start with an empty conversation history.
        message_history = []

    # Start an infinite loop to continuously interact with the user.
    while True:
        # Prompt the user for input and store it in a variable.
        prompt = input('User: >>> ')
        # Check if the user wants to stop the chat by typing 'STOP', case-insensitive.
        if prompt.strip().upper() == 'STOP':
            # If the user types 'STOP', print a goodbye message and exit the loop to end the chat.
            print("Bot: Ciao Ciao!")
            break

        print(message_history)
        # If the user doesn't want to stop, generate the bot's reply using the conversation history and user prompt.
        message_history, reply = chatbot_reply(message_history, prompt)
        # Print the chatbot's reply to the console.

        print(f"ChatGPT: >>> {reply}")

In [ ]:
# Run the chatbot with an optional system prompt to provide context or instructions.
run_chatgpt_chatbot(system_prompt='Act as a friendly assistant', history_window=10, temperature=0.3)

Hello! I am your friendly chatbot. Let's chat! (type 'STOP' to end)
User: >>> STOP
Bot: Ciao Ciao!


In [ ]:
run_chatgpt_chatbot('Act as a sarcastic child', history_window=50, temperature=0.5)

Hello! I am your friendly chatbot. Let's chat! (type 'STOP' to end)
User: >>> STOP
Bot: Ciao Ciao!


In [ ]:
# prompt for outline expansion
#Act as an outline expander.
# Generate a bullet point outline based on the input that I give you
# and then ask me for which bullet point you should expand on.
# Each bullet can have at most 3-5 sub bullets.
# The bullets should be numbered using the pattern [i-v].
# Create a new outline for the bullet point that I select.
# At the end, ask me for what bullet point to expand next.
# Ask me for what to outline.


# The outline should be about purchasing a house.
# I know that I need to perform steps like selecting a location,
# looking at various houses, house features, make an offer etc.
# Provide a complete sequence of steps for me. Fill in any missing steps.



run_chatgpt_chatbot('Act as a helpful assistant who knows about real estate', history_window=50, temperature=0.5)

Hello! I am your friendly chatbot. Let's chat! (type 'STOP' to end)
User: >>> STOP
Bot: Ciao Ciao!


## Build an Interactive UI-based Chatbot using OPEN AI and Gradio

---



- This chatbot takes your text input
- It stores messages in a conversation history
- Uses this history to get responses from ChatGPT
- Shows the reply back to you as a chatbot
- We build a basic UI for this chatbot using Gradio

### Install UI dependencies

In [ ]:
!pip install -q gradio

### Build UI-based ChatGPT Chatbot

In [ ]:
import gradio as gr
import time

In [ ]:
# Define your system prompt directly in the code
system_prompt = "Act as a greedy and sarcastic salesman."

# Define a function to handle the chatbot's reply logic.
def chatbot_reply(history, prompt):
    # The first time the function runs, 'history' will be empty, so we can add the system prompt
    # internally without showing it to the user.
    if not history:
        history.append({'role': 'system', 'content': system_prompt})
    # Add the user's prompt to the conversation history
    history = add_user_prompt(history, prompt)
    # Get a response from the chatbot model based on the updated history.
    response = get_completion(history, temperature=0.1)
    # Maintain the chat history with the new response and ensure it does not exceed the history window.
    history = maintain_chatgpt_message_history(messages=response, history=history, history_window=100)


    # Format the messages for display in the chat interface.
    display_msgs = [{'role': msg['role'], 'content': msg['content']}  for msg in history[1:] ]

    # Return the formatted messages for display in the UI, the updated history, and an empty string to clear the message input box.
    return display_msgs, history, ""


# Create a Gradio Blocks interface to define the UI and functionality.
with gr.Blocks() as demo:
    # Initialize a state to keep track of the message history as an empty list.
    state = gr.State([])

    # Create a row in the UI for the chatbot display component.
    with gr.Row():
        # Initialize the Chatbot interface element with a fixed height.
        chatbot = gr.Chatbot(height=300, label="The good salesman Chatbot", type='messages')

    # Create another row in the UI for user input components.
    with gr.Row():
        # Create a column for the textbox where users can type their messages.
        with gr.Column(scale=20):
            msg = gr.Textbox(label="Ask your salesman", placeholder="Send a Message")
        # Create a narrow column for the send button.
        with gr.Column(scale=1):
            btn = gr.Button("⏩")




    # Bind the textbox to the chatbot_reply function, which triggers on submission.
    msg.submit(chatbot_reply, [state, msg], [chatbot, state, msg], queue=False)

    # Bind the button to the chatbot_reply function as well, which triggers on click.
    btn.click(chatbot_reply, [state, msg], [chatbot, state, msg], queue=False)

# Launch the Gradio interface with debugging enabled (disable in production).
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0dc91ce7859d5fc1f1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://0dc91ce7859d5fc1f1.gradio.live
